# V2 — 7.6–7.11 historical process

This test window originally used a synthetic 56-feature experiment. That retired pipeline is intentionally not executable. This notebook remains as the historical entry point and runs the single canonical V2 implementation in `../7.13-7.18/`.

In [7]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

HERE = Path.cwd()
CANONICAL = HERE.parent / '7.13-7.18'
PATTERN_SCRIPT = CANONICAL / 'forecast_v2_pattern.py'
LABELS = CANONICAL / 'output/serpapi_v2_labels_20260716/serpapi_popular_times_weak_labels.csv'
LEGACY_LABELS = CANONICAL / 'output/serpapi_repeat_audit/legacy_cached_baseline.csv'
MODEL_DIR = CANONICAL / 'output/v2_pattern_known_venue_serpapi_20260716'
assert PATTERN_SCRIPT.exists() and LABELS.exists() and LEGACY_LABELS.exists(), 'Missing canonical V2 artifacts'

In [8]:
# Run the canonical known-venue temporal model.
subprocess.run([
    sys.executable, str(PATTERN_SCRIPT), '--labels', str(LABELS),
    '--legacy-labels', str(LEGACY_LABELS), '--output-dir', str(MODEL_DIR),
], cwd=CANONICAL, check=True)

/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/db_utils.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn, params=params)
/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/db_utils.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn, params=params)


               model_name           model_variant                    split_type split  raw_feature_count  feature_count    mae   rmse    r2  mae_improvement_vs_baseline
GradientBoostingRegressor           baseline_time known_venue_temporal_snapshot train                  6             12 14.742 19.306 0.601                          NaN
GradientBoostingRegressor           baseline_time known_venue_temporal_snapshot  test                  6             12 14.766 19.332 0.600                        0.000
GradientBoostingRegressor venue_specific_enriched known_venue_temporal_snapshot train                 15            185 13.929 17.963 0.654                          NaN
GradientBoostingRegressor venue_specific_enriched known_venue_temporal_snapshot  test                 15            185 13.944 17.980 0.654                        0.822


/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/db_utils.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn, params=params)


CompletedProcess(args=['/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/.venv-1/bin/python', '/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/forecast_v2_pattern.py', '--labels', '/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/output/serpapi_v2_labels_20260716/serpapi_popular_times_weak_labels.csv', '--legacy-labels', '/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/output/serpapi_repeat_audit/legacy_cached_baseline.csv', '--output-dir', '/Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/7.13-7.18/output/v2_pattern_known_venue_serpapi_20260716'], returncode=0)

In [9]:
# Inspect the real cohort, test improvement and dynamic output.
labels = pd.read_csv(LABELS, dtype={'venue_id': str})
metrics = pd.read_csv(MODEL_DIR / 'forecast_v2_pattern_model_metrics.csv')
curve = pd.read_csv(MODEL_DIR / 'prediction_curve_v2_pattern.csv')
manifest = json.loads((MODEL_DIR / 'forecast_v2_pattern_manifest.json').read_text())
test_metrics = metrics[metrics['split'].eq('test')]
display(test_metrics)
display(pd.DataFrame([{
    'label_rows': len(labels), 'venues': labels['venue_id'].nunique(),
    'prediction_rows': len(curve), 'prediction_venues': curve['venue_id'].nunique(),
    'baseline_features': len(manifest['baseline_features']),
    'enriched_features': len(manifest['enriched_features']),
    'dynamic_context_features': len(manifest['dynamic_context_columns']),
}]))
display(curve[['baseline_score', 'dynamic_delta', 'predicted_score']].describe())
display(curve['predicted_level'].value_counts().rename_axis('level').reset_index(name='rows'))

,model_name,model_variant,split_type,split,raw_feature_count,feature_count,mae,rmse,r2,mae_improvement_vs_baseline
1,GradientBoostingRegressor,baseline_time,known_venue_temporal_snapshot,test,6,12,14.766,19.332,0.600,0.000
3,GradientBoostingRegressor,venue_specific_enriched,known_venue_temporal_snapshot,test,15,185,13.944,17.980,0.654,0.822


,label_rows,venues,prediction_rows,prediction_venues,baseline_features,enriched_features,dynamic_context_features
0,20145,162,1944,162,6,15,13


,baseline_score,dynamic_delta,predicted_score
count,1944.000000,1944.000000,1944.000000
mean,27.162901,1.040000,28.202901
std,16.180580,0.866248,16.811509
min,0.000000,0.540000,0.540000
25%,16.220000,0.540000,16.760000
50%,26.190000,0.540000,26.730000
75%,36.262500,1.040000,37.737500
max,81.010000,2.540000,83.550000


,level,rows
0,quiet,1191
1,moderate,729
2,busy,24


## Historical interpretation

The earlier synthetic 56-feature results are historical evidence only and are not comparable to this real temporal-snapshot evaluation. The current model uses 15 learned known-venue inputs, 13 live-context inputs, and reports its improvement against the same temporal baseline.